In [1]:
import pandas as pd
import numpy as np
import os
from coniferest.pineforest import PineForest
from coniferest.session import Session
from coniferest.session.callback import (TerminateAfter, prompt_decision_callback,)
from coniferest.session.callback import (TerminateAfter, viewer_decision_callback,)

In [2]:
from tqdm import tqdm

def load_single(oid_filename, feature_filename):
    oid     = np.memmap(oid_filename, mode='c', dtype=np.uint64)
    feature = np.memmap(feature_filename, mode='c', dtype=np.float32).reshape(oid.shape[0], -1)
    return oid, feature

In [3]:
def group_sorted_oids_features(idx_sort, oids, features, output_dir='fields', batch_size=100000):
    
    os.makedirs(output_dir, exist_ok=True)

    # Check that OIDs and features have the same length
    if len(oids) != len(features):
        raise ValueError("OIDs and features must have the same length.")

    current_field = None
    field_oids = []
    field_features = []

    # Get the number of features per object
    feature_dim = features.shape[1] if features.ndim > 1 else 1
    
    for i in tqdm(idx_sort, desc="Processing OID's"):
        oid = oids[i]
        feature = features[i]
        
        # Determine the field from the first 3 or 4 digits of the OID
        oid_str = str(oid)
        field_len = 4 if len(oid_str) == 16 else 3
        field = oid_str[:field_len]
        
        # Save the current batch if:
        # 1) the field has changed, or
        # 2) the batch is full
        if field != current_field or len(field_oids) >= batch_size:
            if field_oids:  
                save_field_batch(current_field, field_oids, field_features, output_dir, feature_dim)
                field_oids, field_features = [], []
            current_field = field
        
        # Add the current object to the batch
        field_oids.append(oid)
        field_features.append(feature)
    
    # Save the last remaining batch
    if field_oids:
        save_field_batch(current_field, field_oids, field_features, output_dir, feature_dim)

In [4]:
fields = [686, 769]

In [5]:
all_oids = []
all_features = []

for field in fields:
    oid_filename = f"/media2/SNAD/dr23-features-collected_by_field/dr23_oid_{field}.dat"
    feature_filename = f"/media2/SNAD/dr23-features-collected_by_field/dr23_feature_{field}.dat"

    oids_field, features_field = load_single(oid_filename, feature_filename)

    all_oids.append(np.array(oids_field))
    all_features.append(np.array(features_field))

oids_use = np.concatenate(all_oids)
features_use = np.concatenate(all_features)
oids_use


array([686201100000000, 686201100000001, 686201100000002, ...,
       769216400076366, 769216400076368, 769216400076369],
      shape=(13635400,), dtype=uint64)

In [6]:
oids_use.shape


(13635400,)

In [7]:
features_use.shape

(13635400, 47)

In [8]:
data = features_use
metadata = oids_use

In [20]:
model_ztf = PineForest(
    # Number of trees to use for predictions
    n_trees=256,
    # Number of new tree to grow for each decision
    n_spare_trees=768,
)

class RecordCallback:
    def __init__(self):
        self.records = []

    def __call__(self, metadata, data, session):
        decision = session.last_decision
        
        if decision == 1:
            label = "REGULAR"
        elif decision == -1:
            label = "ANOMALY"
        else:
            label = "UNKNOWN"
        self.records.append(f'{metadata} -> {label}')

    def print_report(self):
        print('Records:')
        print('\n'.join(self.records))
        
record_callback = RecordCallback()

session_ztf = Session(
    data=data,
    metadata=metadata,
    model=model_ztf,
    # Prompt for a decision and open object's page on the SNAD Viewer
    decision_callback=viewer_decision_callback,
    on_decision_callbacks=[
        record_callback,
        TerminateAfter(4),
    ]
)
session_ztf.run()
record_callback.print_report()

Check https://ztf.snad.space/view/686212100039979 for details
Is 686212100039979 an anomaly? ([A]nomaly / yes, [R]egular / no, [U]nknown):

  U


Check https://ztf.snad.space/view/686210100013764 for details
Is 686210100013764 an anomaly? ([A]nomaly / yes, [R]egular / no, [U]nknown):

  R


Check https://ztf.snad.space/view/769208300102988 for details
Is 769208300102988 an anomaly? ([A]nomaly / yes, [R]egular / no, [U]nknown):

  R


Check https://ztf.snad.space/view/686216300031065 for details
Is 686216300031065 an anomaly? ([A]nomaly / yes, [R]egular / no, [U]nknown):

  R


Records:
686212100039979 -> UNKNOWN
686210100013764 -> REGULAR
769208300102988 -> REGULAR
686216300031065 -> REGULAR



Records:

